# gatenet on Colab

Train `gatenet.py` on Colab's GPU instead of Claire's laptop, off the crop cache built by
`packcrops.py`. Nothing here reimplements the model, the loss, the split, the eval or the
report: it imports `gatenet.py` and calls `gatenet.train()`, so a number produced here is
directly comparable to the tables already in `TRAINING.md`. The only thing swapped out is
where the pixels come from.

**Why this notebook exists, measured rather than assumed.** The laptop's epochs ran 27.6 s
for the first few and **72.5 s median over 190 epochs** once the i7-8650U dropped off turbo
at epoch 29, and the obvious diagnosis -- JPEG decode -- turned out to be wrong. With the
crop cache the epoch is still 72 s, because the same optimiser step on *synthetic batches
that never touch the dataset* takes **64.3 s per epoch** on the MX150. The laptop is
GPU-bound at ~98%, so the cache buys no speed there.

What it buys is exactly this notebook: the tiles are 4.4 GB and self-contained, so training
can move to a GPU that is not sharing a 15 W thermal budget with the simulator Claire needs
to fly. Moving 200k JPEGs to Drive was never an option.

**Prediction, with a kill criterion.** A T4 should put the GPU floor near 8 s/epoch. At that
point the loader becomes the limit: measured on the laptop, one train-epoch pass over the
loader alone is 13-15 s cached (18-21 s uncached) at 4 workers. So expect roughly
**15-25 s/epoch here, not 8**. If cell 7 reports epochs well above ~30 s the loader is the
bottleneck, and the fix is to hold the tiles in RAM (`np.array` the memmap once -- 4.4 GB
fits in a high-RAM runtime) or raise `WORKERS`, *not* to change anything in `packcrops.py`.

---

## WHAT TO UPLOAD, AND WHERE

Build the cache locally first (about 45 s, writes 4.41 GB):

```
python3 pilot/perception/packcrops.py --mode pack
```

Then put this on Drive, in **`MyDrive/vqual2/`**:

```
MyDrive/vqual2/
  perception/
    gatenet.py               <- unchanged, from pilot/perception/
    packcrops.py             <- unchanged, from pilot/perception/
    autolabel.py             <- gatenet imports canon() from it
    label.py                 <- autolabel imports it
    autolabels_vq1.json      <- 3.9 MB; only read for the staleness fingerprint
  crop_cache/
    tiles.npy                <- 4.41 GB, from pilot/perception/crop_cache/
    index.npz
    meta.json
  gatenet_runs/              <- created for you; checkpoints and the report land here
```

Upload `tiles.npy` with the Drive **desktop client or drive.google.com**, not from a Colab
cell -- a 4.4 GB browser upload into `/content` is thrown away when the session dies.

Frames are **not** needed: nothing in this notebook opens a JPEG. Neither is a checkpoint --
set `RESUME = True` below to continue from whatever is already in
`gatenet_runs/<TAG>/last.pt` on Drive.

If you keep the files somewhere else, change `DRIVE` in cell 3 and nothing else.


## 1. What did Colab actually give us?

Colab varies by the hour: T4 / L4 / A100, 2-12 vCPU, 12-85 GB RAM. Every decision below
(batch size, worker count, whether the cache fits in RAM) depends on it, so measure rather
than assume.

In [ ]:
import os, subprocess, sys, time, shutil, json

print("--- GPU " + "-" * 60)
try:
    print(subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"], text=True).strip())
except Exception as e:
    print("no nvidia-smi:", e)

import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"  {p.name}  {p.total_memory/2**30:.1f} GiB  sm_{p.major}{p.minor}  "
          f"{p.multi_processor_count} SMs")

print("--- CPU / RAM " + "-" * 53)
print("vCPU (os.cpu_count):", os.cpu_count(),
      "| affinity:", len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else "?")
mem = {}
for line in open("/proc/meminfo"):
    k, v = line.split(":", 1)
    mem[k] = v.strip()
print("MemTotal:", mem.get("MemTotal"), "| MemAvailable:", mem.get("MemAvailable"))
print("--- DISK (/content) " + "-" * 48)
t, u, f = shutil.disk_usage("/content")
print(f"total {t/2**30:.0f} GiB, free {f/2**30:.0f} GiB")

# The cache is 4.41 GB. If free space here is under ~10 GB the copy below will not fit and
# you want a Colab Pro high-RAM / larger-disk runtime, not a workaround.
assert f > 8 * 2**30, "not enough local disk for the 4.41 GB cache + checkpoints"

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE      = "/content/drive/MyDrive/vqual2"   # <- the one path to change
CODE_DRIVE = f"{DRIVE}/perception"             # gatenet.py, packcrops.py, autolabel.py,
                                               # label.py, autolabels_vq1.json
FRAMES_ZIP = f"{DRIVE}/vq1_frames.zip"         # 247 MB, the 6598 LABELLED frames only
RUNS_DRIVE = f"{DRIVE}/gatenet_runs"

# Everything below lives on Colab's local NVMe, never on the Drive FUSE mount.
CODE_LOCAL  = "/content/code"                  # ROOT for gatenet's path resolution
SESS_LOCAL  = "/content/sessions"
LOCAL_CACHE = "/content/crop_cache"

for p_ in (DRIVE, CODE_DRIVE):
    assert os.path.isdir(p_), f"missing on Drive: {p_}  (see the upload list at the top)"
assert os.path.isfile(FRAMES_ZIP), f"missing on Drive: {FRAMES_ZIP}"
os.makedirs(RUNS_DRIVE, exist_ok=True)
print("Drive OK. frames zip", f"{os.path.getsize(FRAMES_ZIP)/1e6:.0f} MB")


## 3. Unzip the frames and REBUILD the cache here

The cache is 4.41 GB but the frames it is built from are 247 MB, and `packcrops.py` rebuilds
it byte-identically in well under a minute locally. So we upload the small thing and rebuild
the big thing on Colab's NVMe -- which also removes the multi-minute Drive to `/content` copy
that would otherwise repeat on every reconnect.

`gatenet.py` resolves frame paths as `ROOT/pilot/sessions/<session>/frames/<file>`, where
`ROOT` is two directories above `gatenet.py`. So we copy the code to `/content/code/pilot/
perception/` and symlink `sessions` next to it -- no monkeypatching, and a subprocess call
resolves paths the same way the local machine does.


In [ ]:
import zipfile

# code off Drive onto local disk, so ROOT resolves locally and imports are not FUSE reads
os.makedirs(f"{CODE_LOCAL}/pilot", exist_ok=True)
shutil.rmtree(f"{CODE_LOCAL}/pilot/perception", ignore_errors=True)
shutil.copytree(CODE_DRIVE, f"{CODE_LOCAL}/pilot/perception")

# frames
if not os.path.isdir(SESS_LOCAL):
    t0 = time.time()
    os.makedirs(SESS_LOCAL, exist_ok=True)
    with zipfile.ZipFile(FRAMES_ZIP) as z:
        n = len(z.namelist())
        z.extractall(SESS_LOCAL)
    print(f"unzipped {n} frames in {time.time()-t0:.0f} s")
else:
    print("frames already unzipped")

link = f"{CODE_LOCAL}/pilot/sessions"
if not os.path.islink(link) and not os.path.isdir(link):
    os.symlink(SESS_LOCAL, link)

# rebuild the cache. --mode pack is deterministic; the meta.json fingerprint below must
# match the local one, or the tiles are not the tiles TRAINING.md was measured on.
t0 = time.time()
r = subprocess.run([sys.executable, f"{CODE_LOCAL}/pilot/perception/packcrops.py",
                    "--mode", "pack", "--out", LOCAL_CACHE],
                   capture_output=True, text=True)
print(r.stdout[-2000:] or r.stderr[-2000:])
assert r.returncode == 0, "pack failed -- read the output above"
print(f"cache rebuilt in {time.time()-t0:.0f} s")
print(json.dumps(json.load(open(f"{LOCAL_CACHE}/meta.json")), indent=1))


## 4. Import gatenet unchanged and point it at the cache

`packcrops.attach()` replaces exactly two functions — `gatenet.load_index` (read the
sidecar index instead of the label JSON) and `gatenet.make_loaders` (read tiles instead of
JPEGs). The model, loss, optimiser, cosine schedule, checkpoint format, `--resume`,
`evaluate()` and `breakdown()` are the originals.

`attach()` also refuses a cache whose fingerprint (label-file sha256, `RES`, `MARGIN`,
`MIN_VIS_PX`, frame size) does not match the code it is being used with. A stale cache is
the failure mode that would silently produce a comparable-looking table that is not
comparable, so it is a hard error, not a warning.

In [ ]:
sys.path.insert(0, f"{CODE_LOCAL}/pilot/perception")
import gatenet as G
import packcrops as P

items, tiles, meta = P.attach(LOCAL_CACHE)
print(f"instances: {len(items)}   tile {meta['tile_res']}x{meta['tile_res']} "
      f"scale {meta['tile_scale']}   eval_exact={meta['eval_exact']}")
print(f"RES={G.RES} MARGIN={G.MARGIN} MIN_VIS_PX={G.MIN_VIS_PX} "
      f"BLOCK={G.BLOCK} GUARD={G.GUARD}")

tr_items, va_items = G.split_index(items, "block")
print(f"block split: train {len(tr_items)} / val {len(va_items)}   "
      f"(TRAINING.md says 9406 / 2670)")


### 4b. Sanity: the tiles are pictures of gates, in the right frame

Two minutes here beats a night of training a frame bug. This is `gatenet.sanity()`'s check
done against the cache: green = ground-truth corners drawn from the target the loader will
hand the net, yellow dot = corner 0 (the `min(x+y)` corner, the winding convention). If the
quads do not sit on the apertures, or the yellow dot wanders between tiles, stop.

In [ ]:
import numpy as np, cv2
from matplotlib import pyplot as plt

ds = P.CachedGateCrops(va_items, LOCAL_CACHE, meta, train=False)
sel = np.linspace(0, len(ds) - 1, 8).astype(int)
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
for a, k in zip(ax.ravel(), sel):
    x, y, g, _ = ds[int(k)]
    img = ((x * 0.25 + 0.45) * 255).clamp(0, 255).byte().numpy().transpose(1, 2, 0)
    img = np.ascontiguousarray(img[:, :, ::-1])          # BGR -> RGB for matplotlib
    q = (y.numpy().reshape(4, 2) + 1.0) * (G.RES / 2.0)
    cv2.polylines(img, [q.astype(np.int32).reshape(-1, 1, 2)], True, (0, 255, 0), 1)
    cv2.circle(img, tuple(q[0].astype(int)), 4, (255, 255, 0), -1)
    a.imshow(img); a.set_title(f"{va_items[int(k)]['size_px']:.0f} px"
                               + (" CLIP" if va_items[int(k)]['clipped'] else ""))
    a.axis("off")
plt.tight_layout(); plt.show()

## 5. Train

`gatenet.train()` writes `last.pt` **and** the metrics CSV every epoch and `best.pt`
whenever the median corner error improves. Pointing `G.RUNS` at Drive means a Colab
disconnect costs at most the epoch in flight; re-run this cell with `RESUME = True` and it
picks up from `last.pt`, restoring optimiser, scaler, RNG, epoch and best-so-far.

`EPOCHS` on a resume deliberately reshapes the cosine — that behaviour is `gatenet.py`'s and
is documented there: stopping a run at high LR with the anneal never applied understates the
model, which is the one error this exercise must not make.

Set `TAG` to something that says where it ran. A cached Colab run and the laptop run are
comparable by construction, but only if you can tell them apart afterwards.

In [ ]:
import types, numpy as np, cv2

TAG = "colab-block"
SPLIT = "block"          # "block" or "session" -- both defined in gatenet.py
EPOCHS = 200
RESUME = False           # flip to True after a disconnect and re-run this cell
BATCH = 256              # laptop used 64 on an MX150; a T4/L4/A100 wants more
WORKERS = min(8, os.cpu_count() or 2)
MAX_HOURS = 3.0

# Checkpoints and the appended report go to DRIVE so a disconnect cannot lose them.
G.RUNS = RUNS_DRIVE
G.REPORT = f"{DRIVE}/TRAINING_colab.md"

args = types.SimpleNamespace(
    mode="train", split=SPLIT, tag=TAG, epochs=EPOCHS, batch=BATCH, lr=3e-4,
    width=1.0, workers=WORKERS, max_hours=MAX_HOURS, resume=RESUME, hflip=False,
    ckpt="best", cpu=False, res=G.RES, seed=0)

torch.manual_seed(args.seed); np.random.seed(args.seed)
cv2.setNumThreads(0); torch.backends.cudnn.benchmark = True

t0 = time.time()
rows = G.train(args)
print(f"\nwall clock {(time.time()-t0)/60:.1f} min")

## 6. The report, in `TRAINING.md`'s format

`gatenet.train()` already appended this to `G.REPORT` on Drive. Reprinting it here from the
best checkpoint so the notebook output stands on its own, and re-running the eval is the
cheap way to confirm the checkpoint on Drive is the one that produced the numbers.

Compare against `TRAINING.md`'s `block` run: **ALL corner med 0.89 px, p90 7.15, centre med
0.76**. The cache is measured to move a table cell by at most 0.049 px (`packcrops.py
--mode verify`), so a difference bigger than that is the training run, not the cache.

In [ ]:
from torch.utils.data import DataLoader

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
m = G.GateNet(1.0).to(dev)
ck = torch.load(f"{RUNS_DRIVE}/{TAG}/best.pt", map_location=dev, weights_only=False)
m.load_state_dict(ck["model"]); m.eval()
print(f"best.pt: epoch {ck['epoch']}, best val corner med {ck['best']:.3f} px, "
      f"{ck['nparam']:,} params")

va = DataLoader(P.CachedGateCrops(va_items, LOCAL_CACHE, meta, train=False),
                batch_size=256, shuffle=False, num_workers=WORKERS)
res = G.evaluate(m, va, dev, va_items)
print()
print(G.fmt_rows(G.breakdown(res, va_items)))
print()
print(G.bench_str(m, dev))

## 7. Bring the result home

The checkpoints are already on Drive. Copy `gatenet_runs/<TAG>/` back into
`pilot/perception/gatenet_runs/` on the laptop and paste the table above into
`TRAINING.md` under a run heading that names Colab and the GPU printed in cell 1 — the
hardware is part of the measurement, and "2.6 h for 128 epochs" means nothing without it.

In [ ]:
print("on Drive:", RUNS_DRIVE + "/" + TAG)
for f in sorted(os.listdir(f"{RUNS_DRIVE}/{TAG}")):
    print(f"  {f}  {os.path.getsize(f'{RUNS_DRIVE}/{TAG}/{f}')/1e6:.1f} MB")
print("\nper-epoch times (s) from log.csv:")
import csv as _csv
rows_ = list(_csv.DictReader(open(f"{RUNS_DRIVE}/{TAG}/log.csv")))
secs = [float(r["secs"]) for r in rows_]
if secs:
    print(f"  n={len(secs)}  median {sorted(secs)[len(secs)//2]:.1f}  "
          f"min {min(secs):.1f}  max {max(secs):.1f}")
    print("  laptop baseline, uncached: 72.5 s median over 190 epochs")